In [1]:
import torch
import clip
import numpy as np
import pandas as pd
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [3]:
# Load the 1854 labels from the things dataset.
things = np.loadtxt("data/words.csv", dtype=str, delimiter=",")
labels = clip.tokenize(things).to(device)

In [ ]:
# Process single image
image = preprocess(Image.open("data/input/window.jpg")).unsqueeze(0).to(device)

In [ ]:
li, lt = model(image, labels)
x = li.softmax(dim=-1).detach().numpy()

In [57]:
window = pd.DataFrame({
    "uniqueID":    np.array(["window"]).repeat(x.shape[1]),
    "label":            things[(-x).argsort()].squeeze(),
    "prob":             -np.sort(-x).squeeze()
})

window

,uniqueID,label,prob
0,window,window,6.812845e-01
1,window,windowsill,7.327873e-02
2,window,wall,1.640949e-02
3,window,door,1.127781e-02
4,window,curtain,1.049265e-02
...,...,...,...
1849,window,headband,5.940163e-08
1850,window,ferris wheel,5.360373e-08
1851,window,seaplane,4.499411e-08
1852,window,manatee,9.782262e-09


In [58]:
window.to_csv("data/clip_scores_window.csv")

In [5]:
# Process directory of images

import os
import glob
from scipy import stats

files = sorted(glob.glob(os.path.join("data/input", '*.*')))

In [6]:
entropy = np.zeros(len(files))
data = []

for i, file in enumerate(files):
    print("({}/{})  {}".format(i+1, len(files), file))
    image = preprocess(Image.open(file)).unsqueeze(0).to(device)
    
    with torch.no_grad():
        li, _ = model(image, labels)
        pr = li.softmax(dim=-1).cpu().numpy()
    
    data.append(
        pd.DataFrame({
        "file":    np.array([file]).repeat(pr.shape[1]),
        "label":            things[(-pr).argsort()].squeeze(),
        "prob":             -np.sort(-pr).squeeze()
         })
    )

    entropy[i] = stats.entropy(pr.squeeze())

pd.concat(data)

(1/45)  data/input/window.jpg
(2/45)  data/input/window_1.jpg
(3/45)  data/input/window_2.jpg
(4/45)  data/input/window_3.jpg
(5/45)  data/input/window_4.jpg
(6/45)  data/input/window_5.jpg
(7/45)  data/input/window_6.jpg
(8/45)  data/input/window_7.jpg
(9/45)  data/input/window_8.jpg
(10/45)  data/input/wire.jpg
(11/45)  data/input/wire_1.jpg
(12/45)  data/input/wire_2.jpg
(13/45)  data/input/wire_3.jpg
(14/45)  data/input/wire_4.jpg
(15/45)  data/input/wire_5.jpg
(16/45)  data/input/wire_6.jpg
(17/45)  data/input/wire_7.jpg
(18/45)  data/input/wire_8.jpg
(19/45)  data/input/wrist.jpg
(20/45)  data/input/wrist_1.jpg
(21/45)  data/input/wrist_2.jpg
(22/45)  data/input/wrist_3.jpg
(23/45)  data/input/wrist_4.jpg
(24/45)  data/input/wrist_5.jpg
(25/45)  data/input/wrist_6.jpg
(26/45)  data/input/wrist_7.jpg
(27/45)  data/input/wrist_8.jpg
(28/45)  data/input/yarn.jpg
(29/45)  data/input/yarn_1.jpg
(30/45)  data/input/yarn_2.jpg
(31/45)  data/input/yarn_3.jpg
(32/45)  data/input/yarn_4.jp

,file,label,prob
0,data/input/window.jpg,window,6.806641e-01
1,data/input/window.jpg,windowsill,7.287598e-02
2,data/input/window.jpg,wall,1.676941e-02
3,data/input/window.jpg,door,1.135254e-02
4,data/input/window.jpg,curtain,1.049805e-02
...,...,...,...
1849,data/input/yolk_8.jpg,manatee,1.788139e-07
1850,data/input/yolk_8.jpg,stockings,1.788139e-07
1851,data/input/yolk_8.jpg,scarf,1.192093e-07
1852,data/input/yolk_8.jpg,flamingo,1.192093e-07


In [7]:
pd.DataFrame({"file": files, "entropy": entropy}).to_csv("data/clip_scores.csv")
pd.concat(data).to_csv("data/clip_full.csv")